# ElevateLabs AI/ML Internship Project Phase
## Project #8: Movie Recommendation System with Hybrid ML & NLP
**Domain:** Artificial Intelligence & Machine Learning  
**Dataset:** MovieLens 100K Benchmark Dataset (GroupLens Research)  
**Tools:** Python, Scikit-Learn, Pandas, NumPy, Matplotlib, Seaborn, Streamlit
---
### Project Objectives:
1. Load, explore, and preprocess MovieLens dataset (movies, ratings, tags).
2. Build Content-Based Filtering using TF-IDF vectorization and Cosine Similarity on genres and tags.
3. Build Collaborative Filtering using TruncatedSVD Matrix Factorization on user-item interaction matrix.
4. Evaluate Collaborative Filtering performance using Root Mean Squared Error (RMSE) and MAE on test split.
5. Implement a Hybrid Recommendation Engine with Bayesian IMDb rating priors to handle cold-start.
6. Test recommendations across iconic films and evaluate accuracy.

In [1]:
# 1. Imports and Setup
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment initialized successfully!')

---
### 2. Loading the Raw Datasets

In [2]:
movies = pd.read_csv('data/ml-latest-small/movies.csv')
ratings = pd.read_csv('data/ml-latest-small/ratings.csv')
tags = pd.read_csv('data/ml-latest-small/tags.csv')

print(f'Movies Dataset Shape:  {movies.shape}')
print(f'Ratings Dataset Shape: {ratings.shape}')
print(f'Tags Dataset Shape:    {tags.shape}')
print(f'Unique Users: {ratings["userId"].nunique()} | Unique Movies: {ratings["movieId"].nunique()}')
display(movies.head(3))
display(ratings.head(3))

---
### 3. Exploratory Data Analysis (EDA) & Visualizations

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating distribution
sns.histplot(ratings['rating'], bins=10, kde=True, color='#e63946', ax=axes[0])
axes[0].set_title('Distribution of User Ratings (0.5 to 5.0 Stars)', fontsize=12)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')

# Most reviewed movies
top_rated_count = ratings.groupby('movieId').size().reset_index(name='vote_count')
top_rated_count = top_rated_count.merge(movies, on='movieId').sort_values('vote_count', ascending=False).head(10)
sns.barplot(data=top_rated_count, x='vote_count', y='title', hue='title', palette='viridis', legend=False, ax=axes[1])
axes[1].set_title('Top 10 Most Reviewed Movies in MovieLens', fontsize=12)
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

---
### 4. Feature Engineering & Preprocessing
- Extract release year via Regex
- Aggregate user tags per movie
- Formulate Metadata Soup (Genres + Tags)
- Compute Bayesian Weighted Rating (IMDb Formula):
$$WR = \left(\frac{v}{v + m}\right) \cdot R + \left(\frac{m}{v + m}\right) \cdot C$$

In [4]:
def extract_year(title):
    m = re.search(r'\((\d{4})\)', str(title))
    return int(m.group(1)) if m else np.nan

movies['year'] = movies['title'].apply(extract_year)
movies['genres_str'] = movies['genres'].apply(lambda x: '' if x == '(no genres listed)' else x.replace('|', ' '))

# Aggregate tags
tags_agg = tags.groupby('movieId')['tag'].apply(lambda t: ' '.join(t.dropna().astype(str).str.lower())).reset_index()
movies = movies.merge(tags_agg, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')
movies['soup'] = movies['genres_str'] + ' ' + movies['genres_str'] + ' ' + movies['tag']

# Rating statistics
stats = ratings.groupby('movieId').agg(vote_count=('rating', 'count'), vote_average=('rating', 'mean')).reset_index()
movies = movies.merge(stats, on='movieId', how='left')
movies['vote_count'] = movies['vote_count'].fillna(0).astype(int)
movies['vote_average'] = movies['vote_average'].fillna(0.0).round(2)

# Bayesian Weighted Rating
C = ratings['rating'].mean()
m = movies['vote_count'].quantile(0.70)
movies['weighted_rating'] = movies.apply(
    lambda row: round((row['vote_count'] / (row['vote_count'] + m) * row['vote_average']) + (m / (row['vote_count'] + m) * C), 2)
    if (row['vote_count'] + m) > 0 else 0.0,
    axis=1
)

print('Top 5 Weighted Rated Masterpieces in Dataset:')
display(movies.sort_values('weighted_rating', ascending=False)[['title', 'year', 'vote_count', 'vote_average', 'weighted_rating']].head(5))

---
### 5. Content-Based Filtering Model
Transforming textual descriptors into TF-IDF matrix and calculating pairwise Cosine Similarity.

In [5]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(movies['soup'].fillna(''))
content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')
print(f'Content Similarity Matrix: {content_sim.shape}')

---
### 6. Collaborative Filtering (TruncatedSVD Matrix Factorization)
Evaluating on an 80/20 train-test split of user ratings.

In [6]:
train_r, test_r = train_test_split(ratings, test_size=0.2, random_state=42)
user_item = train_r.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Truncated SVD
svd = TruncatedSVD(n_components=30, random_state=42)
u_factors = svd.fit_transform(user_item.values)
m_factors = svd.components_
pred_matrix = pd.DataFrame(np.dot(u_factors, m_factors), index=user_item.index, columns=user_item.columns)

y_true, y_pred = [], []
for _, row in test_r.iterrows():
    u, m = row['userId'], row['movieId']
    if u in pred_matrix.index and m in pred_matrix.columns:
        p = max(0.5, min(5.0, pred_matrix.loc[u, m]))
        y_true.append(row['rating'])
        y_pred.append(p)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

print(f'Evaluated on {len(y_true)} test user ratings')
print(f'Collaborative SVD Test RMSE: {rmse:.4f}')
print(f'Collaborative SVD Test MAE:  {mae:.4f}')

---
### 7. Hybrid Recommendation Engine Demonstration
Combining Content-Based Similarity, Collaborative SVD Latent Features, and Bayesian Popularity Prior.

In [7]:
from src.recommender import MovieRecommenderEngine
engine = MovieRecommenderEngine.load_from_artifacts()
if engine is None:
    engine = MovieRecommenderEngine(movies, ratings)
    engine.fit_all()

test_movies = ['Toy Story', 'Inception', 'The Dark Knight']
for tm in test_movies:
    print(f'\n{"="*25} Recommendations for {tm} {"="*25}')
    recs, target = engine.recommend_hybrid(tm, top_n=5)
    if target is not None:
        print(f'Input: {target["title"]} | Genres: {target["genres"]}')
        display(recs[['title', 'year', 'genres', 'vote_average', 'match_score']])
    else:
        print('Movie not found.')

---
### 8. Conclusion & Deliverables
- Successfully built and validated Hybrid Recommendation Model.
- Solved cold-start problem using Content-based Fallbacks and Bayesian Average.
- Produced interactive web dashboard via Streamlit (`app.py`).
- Prepared 2-page project report PDF (`Movie_Recommendation_System_Project_Report.pdf`).